In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import joblib

## 1. Load Datasets

Load the features and labels from the HDF5 files.

In [2]:
# Load the datasets
df_features = pd.read_hdf('features.h5', key='features')
df_labels = pd.read_hdf('labels.h5', key='labels')

print("Features dataframe shape:", df_features.shape)
print("Labels dataframe shape:", df_labels.shape)

Features dataframe shape: (7005279, 140)
Labels dataframe shape: (7005279, 4)


## 2. Prepare Data for Modeling

Select the features (X) and the target variables (y). We will train two separate models: one for `scores` and one for `concedes`.

In [3]:
# Define features (X) and targets (y)
features_to_exclude = ['game_id']
X = df_features.drop(columns=features_to_exclude)
y_scores = df_labels['scores']
y_concedes = df_labels['concedes']

# Split data for the 'scores' model
X_train_scores, X_test_scores, y_train_scores, y_test_scores = train_test_split(
    X, y_scores, test_size=0.2, random_state=42, stratify=y_scores
)

# Split data for the 'concedes' model
X_train_concedes, X_test_concedes, y_train_concedes, y_test_concedes = train_test_split(
    X, y_concedes, test_size=0.2, random_state=42, stratify=y_concedes
)

## 3. Train and Evaluate 'Scores' Model

Train a `RandomForestClassifier` to predict whether a team will score.

In [4]:
# Initialize and train the classifier for scores
clf_scores = RandomForestClassifier(n_estimators=10, random_state=42, class_weight='balanced')
clf_scores.fit(X_train_scores, y_train_scores)

# Make predictions
y_pred_scores = clf_scores.predict(X_test_scores)
y_pred_proba_scores = clf_scores.predict_proba(X_test_scores)[:, 1]

# Evaluate the model
print("--- Scores Model Evaluation ---")
print("Accuracy:", accuracy_score(y_test_scores, y_pred_scores))
print("ROC AUC Score:", roc_auc_score(y_test_scores, y_pred_proba_scores))
print("\nClassification Report:\n", classification_report(y_test_scores, y_pred_scores))

# Save the model
joblib.dump(clf_scores, 'model_scores.joblib')

--- Scores Model Evaluation ---
Accuracy: 0.9899789872781674
ROC AUC Score: 0.6464853579600249

Classification Report:
               precision    recall  f1-score   support

       False       0.99      1.00      0.99   1385084
        True       0.96      0.13      0.22     15972

    accuracy                           0.99   1401056
   macro avg       0.98      0.56      0.61   1401056
weighted avg       0.99      0.99      0.99   1401056



['model_scores.joblib']

## 4. Train and Evaluate 'Concedes' Model

Train a `RandomForestClassifier` to predict whether a team will concede a goal.

In [5]:
# Initialize and train the classifier for concedes
clf_concedes = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf_concedes.fit(X_train_concedes, y_train_concedes)

# Make predictions
y_pred_concedes = clf_concedes.predict(X_test_concedes)
y_pred_proba_concedes = clf_concedes.predict_proba(X_test_concedes)[:, 1]

# Evaluate the model
print("--- Concedes Model Evaluation ---")
print("Accuracy:", accuracy_score(y_test_concedes, y_pred_concedes))
print("ROC AUC Score:", roc_auc_score(y_test_concedes, y_pred_proba_concedes))
print("\nClassification Report:\n", classification_report(y_test_concedes, y_pred_concedes))

# Save the model
joblib.dump(clf_concedes, 'model_concedes.joblib')

--- Concedes Model Evaluation ---
Accuracy: 0.9977038764304379
ROC AUC Score: 0.6861238406483277

Classification Report:
               precision    recall  f1-score   support

       False       1.00      1.00      1.00   1397715
        True       0.79      0.05      0.10      3342

    accuracy                           1.00   1401057
   macro avg       0.90      0.53      0.55   1401057
weighted avg       1.00      1.00      1.00   1401057



['model_concedes.joblib']

## 5. Conclusion

The models for predicting scores and concedes have been trained and evaluated. The trained models are saved as `model_scores.joblib` and `model_concedes.joblib` for future use. You can now use these models to make predictions on new game data.